# Modeling IHSG — Dual-Branch Multivariate LSTM (v2)
## Analisis Prediksi IHSG Menggunakan Multivariate LSTM Berdasarkan Data Historis dan Variabel Makroekonomi Periode 2015–2024

**Pipeline:**
1. Load artefak dari `data/processed_v2/`
2. Definisi arsitektur (Dual-Branch & Univariate)
3. Grid search proxy (5 epoch) → best hyperparameter arsitektur
4. Ablation study `seq_monthly` ∈ {3, 6, 12} dengan full training → pilih seq_monthly terbaik
5. Final training multi-seed (5 seed) → pilih seed terbaik per model
6. Evaluasi test set: RMSE, MAPE, R²
7. Simpan artefak ke `model/model_multivariate_v2/`

**Perbaikan dari v1:**
- Path ke `data/processed_v2/` (y_test sudah konsisten & fix bug sequence)
- Ablation study `seq_monthly` yang proper — tidak hardcode, dipilih dari eksperimen
- Multi-seed final training (5 seed) untuk stabilitas konvergensi
- Output ke `model/model_multivariate_v2/`


## 1. Imports & Konfigurasi

In [1]:
import os
import gc
import json
import warnings
import itertools
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from pathlib import Path
from datetime import datetime
from sklearn.metrics import mean_squared_error, r2_score

from tensorflow.keras import Input, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')

print(f"TensorFlow  : {tf.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"GPU devices : {tf.config.list_physical_devices('GPU') or 'None (CPU only)'}")


TensorFlow  : 2.20.0
NumPy       : 2.2.6
Pandas      : 2.3.3
GPU devices : None (CPU only)


In [2]:
ROOT           = Path(os.getcwd()).parent
DATA_PROCESSED = ROOT / "data" / "processed_v2"       # ← v2
MODEL_DIR      = ROOT / "model" / "model_multivariate_v2"  # ← v2
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT          :", ROOT)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("MODEL_DIR     :", MODEL_DIR)
assert DATA_PROCESSED.exists(), f"Folder tidak ditemukan: {DATA_PROCESSED}"


ROOT          : d:\project_skripsi
DATA_PROCESSED: d:\project_skripsi\data\processed_v2
MODEL_DIR     : d:\project_skripsi\model\model_multivariate_v2


## 2. Memuat Artefak Preprocessing (v2)

Sekuens, scaler, dan metadata hasil tahap preprocessing dimuat dari `data/processed_v2/` agar seluruh percobaan berjalan di atas data yang sama dan konsisten.

In [3]:
with open(DATA_PROCESSED / "preprocessing_meta.json") as f:
    meta = json.load(f)

GRID_CONFIG      = meta['grid_config']
DAILY_FEATURES   = meta['daily_features']
MONTHLY_FEATURES = meta['monthly_features']
TARGET_COL       = meta['target_col']
TARGET_IDX       = meta['target_idx_in_daily']

print(f"Preprocessing version  : {meta['version']}")
print(f"Note                   : {meta['note']}")
print(f"Daily  train/test      : {meta['n_daily_train']} / {meta['n_daily_test']} hari")
print(f"Monthly train/test     : {meta['n_monthly_train']} / {meta['n_monthly_test']} bulan")
print(f"Cutoff date            : {meta['cutoff_date']}")
print(f"Daily features         : {DAILY_FEATURES}")
print(f"Monthly features       : {MONTHLY_FEATURES}")
print(f"Target                 : {TARGET_COL}")
print(f"Grid config            : {GRID_CONFIG}")


Preprocessing version  : v2
Note                   : y_test di-generate canonical (independen dari seq_monthly)
Daily  train/test      : 1940 / 486 hari
Monthly train/test     : 95 / 25 bulan
Cutoff date            : 2022-12-21
Daily features         : ['ihsg', 'nilai_tukar']
Monthly features       : ['inflasi', 'bi_rate']
Target                 : ihsg
Grid config            : {'seq_daily_candidates': [22, 60, 90], 'seq_monthly_candidates': [3, 6, 12], 'batch_size_candidates': [16, 32], 'grid_horizon': 5, 'test_ratio': 0.2, 'random_seed': 42}


In [4]:
scaler_daily   = joblib.load(DATA_PROCESSED / "scaler_daily.joblib")
scaler_monthly = joblib.load(DATA_PROCESSED / "scaler_monthly.joblib")
scaler_target  = joblib.load(DATA_PROCESSED / "scaler_target.joblib")
scaler_uni     = joblib.load(DATA_PROCESSED / "scaler_univariate.joblib")

sequences_dual = joblib.load(DATA_PROCESSED / "sequences_dual.joblib")
sequences_uni  = joblib.load(DATA_PROCESSED / "sequences_uni.joblib")

print(f"scaler_target IHSG train range: [{scaler_target.data_min_[0]:.0f}, {scaler_target.data_max_[0]:.0f}]")
print(f"\nsequences_dual: {len(sequences_dual)} kombinasi")
print(f"{'Key':<12} {'X_daily_te':>12} {'X_monthly_te':>14} {'y_test':>10} {'y_range':>20}")
print("-" * 72)
for k, v in sequences_dual.items():
    y = v['y_test']
    print(f"{str(k):<12} {str(v['X_daily_test'].shape):>12} "
          f"{str(v['X_monthly_test'].shape):>14} {str(y.shape):>10} "
          f"[{y.min():.4f}, {y.max():.4f}]")

print(f"\nsequences_uni : {len(sequences_uni)} seq_daily")
for k, v in sequences_uni.items():
    print(f"  seq={k}: X_train={v['X_train'].shape}, y_test range=[{v['y_test'].min():.4f},{v['y_test'].max():.4f}]")


scaler_target IHSG train range: [3938, 7318]

sequences_dual: 9 kombinasi
Key            X_daily_te   X_monthly_te     y_test              y_range
------------------------------------------------------------------------
(22, 3)      (485, 22, 2)    (485, 3, 2)     (485,) [0.7775, 1.1738]
(22, 6)      (485, 22, 2)    (485, 6, 2)     (485,) [0.7775, 1.1738]
(22, 12)     (485, 22, 2)   (485, 12, 2)     (485,) [0.7775, 1.1738]
(60, 3)      (485, 60, 2)    (485, 3, 2)     (485,) [0.7775, 1.1738]
(60, 6)      (485, 60, 2)    (485, 6, 2)     (485,) [0.7775, 1.1738]
(60, 12)     (485, 60, 2)   (485, 12, 2)     (485,) [0.7775, 1.1738]
(90, 3)      (485, 90, 2)    (485, 3, 2)     (485,) [0.7775, 1.1738]
(90, 6)      (485, 90, 2)    (485, 6, 2)     (485,) [0.7775, 1.1738]
(90, 12)     (485, 90, 2)   (485, 12, 2)     (485,) [0.7775, 1.1738]

sequences_uni : 3 seq_daily
  seq=22: X_train=(1917, 22, 1), y_test range=[0.7775,1.1738]
  seq=60: X_train=(1879, 60, 1), y_test range=[0.7775,1.1738]
  seq=

## 3. Konfigurasi Hyperparameter & Training

Mendefinisikan ruang pencarian hyperparameter (grid) beserta parameter training yang dipakai pada tahap-tahap berikutnya.

In [5]:
# ── Grid Search (proxy 5 epoch) ──
# Mencari hyperparameter arsitektur terbaik: lstm_units, dropout_rate, batch_size,
# learning_rate, dan seq_daily. seq_monthly TIDAK diputuskan di grid search
# melainkan melalui ablation study terpisah (Section 5).
GRID = {
    "seq_daily"     : GRID_CONFIG['seq_daily_candidates'],   # [22, 60, 90]
    "seq_monthly"   : GRID_CONFIG['seq_monthly_candidates'], # [3, 6, 12] — hanya untuk grid
    "lstm_units"    : [64, 128],
    "dropout_rate"  : [0.2, 0.3],
    "batch_size"    : GRID_CONFIG['batch_size_candidates'],  # [16, 32]
    "learning_rate" : [0.001, 0.0005],
}

GRID_UNI = {
    "seq_daily"     : GRID_CONFIG['seq_daily_candidates'],
    "lstm_units"    : [64, 128],
    "dropout_rate"  : [0.2, 0.3],
    "batch_size"    : GRID_CONFIG['batch_size_candidates'],
    "learning_rate" : [0.001, 0.0005],
}

GRID_HORIZON = GRID_CONFIG['grid_horizon']  # 5 epoch proxy
FINAL_EPOCHS = 150
PATIENCE_ES  = 20
PATIENCE_LR  = 10
VAL_SPLIT    = 0.15

# ── Multi-seed config ──
# Seeds digunakan untuk stabilitas konvergensi — model terbaik dari N run
SEED_LIST = [42, 7, 13, 21, 99]
N_SEEDS   = len(SEED_LIST)

n_dual = (len(GRID['seq_daily']) * len(GRID['seq_monthly']) *
          len(GRID['lstm_units']) * len(GRID['dropout_rate']) *
          len(GRID['batch_size']) * len(GRID['learning_rate']))
n_uni  = (len(GRID_UNI['seq_daily']) * len(GRID_UNI['lstm_units']) *
          len(GRID_UNI['dropout_rate']) * len(GRID_UNI['batch_size']) *
          len(GRID_UNI['learning_rate']))

print(f"Multivariate grid  : {n_dual} kombinasi × {GRID_HORIZON} epoch proxy")
print(f"Univariate   grid  : {n_uni} kombinasi × {GRID_HORIZON} epoch proxy")
print(f"FINAL_EPOCHS       : {FINAL_EPOCHS} (EarlyStopping patience={PATIENCE_ES})")
print(f"N_SEEDS            : {N_SEEDS}  {SEED_LIST}")
print(f"VAL_SPLIT          : {VAL_SPLIT}")


Multivariate grid  : 144 kombinasi × 5 epoch proxy
Univariate   grid  : 48 kombinasi × 5 epoch proxy
FINAL_EPOCHS       : 150 (EarlyStopping patience=20)
N_SEEDS            : 5  [42, 7, 13, 21, 99]
VAL_SPLIT          : 0.15


## 4. Arsitektur Model

Mendefinisikan dua arsitektur yang dibandingkan: Dual-Branch LSTM (multivariate) dan baseline single-branch (univariate).

In [6]:
def build_lstm_dual_branch(seq_daily, seq_monthly,
                            lstm_units=64, dropout_rate=0.2,
                            learning_rate=0.001,
                            n_daily_features=2, n_monthly_features=2):
    """
    Dual-Branch LSTM — Keras Functional API.
    Branch daily : IHSG + Kurs (frekuensi harian)
    Branch monthly: Inflasi + BI Rate (frekuensi bulanan)
    """
    # Branch Daily
    inp_daily = Input(shape=(seq_daily, n_daily_features), name='input_daily')
    x_d = LSTM(lstm_units, return_sequences=True, name='lstm_daily_1')(inp_daily)
    x_d = Dropout(dropout_rate, name='drop_daily_1')(x_d)
    x_d = LSTM(lstm_units // 2, return_sequences=False, name='lstm_daily_2')(x_d)
    x_d = Dropout(dropout_rate, name='drop_daily_2')(x_d)

    # Branch Monthly
    inp_monthly = Input(shape=(seq_monthly, n_monthly_features), name='input_monthly')
    x_m = LSTM(lstm_units, return_sequences=True, name='lstm_monthly_1')(inp_monthly)
    x_m = Dropout(dropout_rate, name='drop_monthly_1')(x_m)
    x_m = LSTM(lstm_units // 2, return_sequences=False, name='lstm_monthly_2')(x_m)
    x_m = Dropout(dropout_rate, name='drop_monthly_2')(x_m)

    # Merge
    merged = Concatenate(name='concat')([x_d, x_m])
    x      = Dense(lstm_units // 4, activation='relu', name='dense_hidden')(merged)
    x      = Dropout(dropout_rate, name='drop_final')(x)
    output = Dense(1, activation='linear', name='output')(x)

    model = Model(inputs=[inp_daily, inp_monthly], outputs=output,
                  name='LSTM_DualBranch_Multivariate')
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='mse', metrics=['mae'])
    return model


def build_lstm_univariate(seq_daily, lstm_units=64, dropout_rate=0.2,
                           learning_rate=0.001):
    """
    Univariate LSTM — arsitektur identik satu branch dari Dual-Branch.
    Input: IHSG only. Digunakan sebagai baseline perbandingan.
    """
    inp    = Input(shape=(seq_daily, 1), name='input_ihsg')
    x      = LSTM(lstm_units, return_sequences=True, name='lstm_1')(inp)
    x      = Dropout(dropout_rate, name='drop_1')(x)
    x      = LSTM(lstm_units // 2, return_sequences=False, name='lstm_2')(x)
    x      = Dropout(dropout_rate, name='drop_2')(x)
    x      = Dense(lstm_units // 4, activation='relu', name='dense_hidden')(x)
    x      = Dropout(dropout_rate, name='drop_final')(x)
    output = Dense(1, activation='linear', name='output')(x)

    model = Model(inputs=inp, outputs=output, name='LSTM_Univariate')
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='mse', metrics=['mae'])
    return model


def compute_metrics(y_true_scaled, y_pred_scaled, scaler_t):
    """Inverse transform dan hitung RMSE, MAPE, R² dalam skala asli IHSG."""
    y_true = scaler_t.inverse_transform(y_true_scaled.reshape(-1, 1)).flatten()
    y_pred = scaler_t.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    mape   = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    r2     = r2_score(y_true, y_pred)
    return {"rmse": rmse, "mape": mape, "r2": r2, "y_true": y_true, "y_pred": y_pred}


print("✓ build_lstm_dual_branch()")
print("✓ build_lstm_univariate()")
print("✓ compute_metrics()")


✓ build_lstm_dual_branch()
✓ build_lstm_univariate()
✓ compute_metrics()


## 5. Grid Search Multivariate (proxy 5 epoch)

Untuk menekan biaya komputasi, tiap kombinasi hyperparameter arsitektur (`lstm_units`, `dropout_rate`, `batch_size`, `learning_rate`, dan `seq_daily`) dievaluasi lewat proxy singkat 5 epoch, lalu kombinasi dengan `val_loss` terendah dipilih sebagai basis arsitektur. Seluruh nilai `seq_monthly` tetap diikutkan pada tahap ini agar cakupan pencarian lebih luas.

> **Catatan metodologi:** nilai `seq_monthly` final tidak ditetapkan di sini, melainkan melalui ablation study tersendiri (Bagian 7). Alasannya, branch bulanan membutuhkan lebih banyak epoch untuk benar-benar berkontribusi, sehingga proxy 5 epoch cenderung meremehkan perannya.

In [7]:
grid_results_multi = []
best_multi         = {"val_loss": np.inf, "config": None}

available_keys  = set(sequences_dual.keys())
hp_keys_seq     = [(sd, sm) for sd in GRID['seq_daily']
                             for sm in GRID['seq_monthly']
                             if (sd, sm) in available_keys]
hp_keys_rest    = list(itertools.product(
    GRID['lstm_units'], GRID['dropout_rate'],
    GRID['batch_size'], GRID['learning_rate']
))
total_multi = len(hp_keys_seq) * len(hp_keys_rest)

print(f"Memulai grid search multivariate: {total_multi} kombinasi × {GRID_HORIZON} epoch")
print("=" * 78)

run_idx    = 0
start_time = datetime.now()

for (seq_d, seq_m) in hp_keys_seq:
    seqs   = sequences_dual[(seq_d, seq_m)]
    X_d_tr = seqs['X_daily_train']
    X_m_tr = seqs['X_monthly_train']
    y_tr   = seqs['y_train']

    for (units, drop, bs, lr) in hp_keys_rest:
        run_idx += 1
        tf.keras.backend.clear_session()
        gc.collect()

        model = build_lstm_dual_branch(
            seq_daily=seq_d, seq_monthly=seq_m,
            lstm_units=units, dropout_rate=drop, learning_rate=lr
        )
        history = model.fit(
            [X_d_tr, X_m_tr], y_tr,
            epochs=GRID_HORIZON, batch_size=bs,
            validation_split=VAL_SPLIT,
            verbose=0, shuffle=False
        )

        val_loss = min(history.history['val_loss'])
        config   = {
            "seq_daily": seq_d, "seq_monthly": seq_m,
            "lstm_units": units, "dropout_rate": drop,
            "batch_size": bs, "learning_rate": lr,
            "val_loss_proxy": round(float(val_loss), 8),
        }
        grid_results_multi.append(config)

        marker = " ◄ BEST" if val_loss < best_multi['val_loss'] else ""
        if val_loss < best_multi['val_loss']:
            best_multi = {"val_loss": val_loss, "config": config}

        elapsed = (datetime.now() - start_time).seconds
        print(f"[{run_idx:3d}/{total_multi}] d={seq_d:2d} m={seq_m:2d} "
              f"u={units:3d} dr={drop} bs={bs:2d} lr={lr} "
              f"→ val_loss={val_loss:.6f}{marker}  ({elapsed}s)")

grid_results_multi_sorted = sorted(grid_results_multi, key=lambda x: x['val_loss_proxy'])
with open(MODEL_DIR / "grid_results_multivariate.json", "w") as f:
    json.dump({"best": best_multi['config'],
               "all_results": grid_results_multi_sorted}, f, indent=2)

print(f"\n✓ Grid search multivariate selesai")
print(f"  Best config   : {best_multi['config']}")
print(f"  Best val_loss : {best_multi['val_loss']:.6f}")
print(f"  Disimpan      → grid_results_multivariate.json")


Memulai grid search multivariate: 144 kombinasi × 5 epoch

[  1/144] d=22 m= 3 u= 64 dr=0.2 bs=16 lr=0.001 → val_loss=0.001928 ◄ BEST  (20s)
[  2/144] d=22 m= 3 u= 64 dr=0.2 bs=16 lr=0.0005 → val_loss=0.002966  (38s)
[  3/144] d=22 m= 3 u= 64 dr=0.2 bs=32 lr=0.001 → val_loss=0.016122  (55s)
[  4/144] d=22 m= 3 u= 64 dr=0.2 bs=32 lr=0.0005 → val_loss=0.010897  (72s)
[  5/144] d=22 m= 3 u= 64 dr=0.3 bs=16 lr=0.001 → val_loss=0.005138  (90s)
[  6/144] d=22 m= 3 u= 64 dr=0.3 bs=16 lr=0.0005 → val_loss=0.007630  (108s)
[  7/144] d=22 m= 3 u= 64 dr=0.3 bs=32 lr=0.001 → val_loss=0.025502  (125s)
[  8/144] d=22 m= 3 u= 64 dr=0.3 bs=32 lr=0.0005 → val_loss=0.018655  (142s)
[  9/144] d=22 m= 3 u=128 dr=0.2 bs=16 lr=0.001 → val_loss=0.005151  (168s)
[ 10/144] d=22 m= 3 u=128 dr=0.2 bs=16 lr=0.0005 → val_loss=0.002281  (192s)
[ 11/144] d=22 m= 3 u=128 dr=0.2 bs=32 lr=0.001 → val_loss=0.003333  (213s)
[ 12/144] d=22 m= 3 u=128 dr=0.2 bs=32 lr=0.0005 → val_loss=0.001533 ◄ BEST  (235s)
[ 13/144] d=22

## 6. Grid Search Univariate (proxy 5 epoch)

Prosedur grid search yang sama diterapkan pada baseline univariate sebagai pembanding, memakai proxy 5 epoch untuk memilih hyperparameter arsitektur terbaiknya.

In [8]:
grid_results_uni = []
best_uni         = {"val_loss": np.inf, "config": None}

hp_uni_combos = list(itertools.product(
    GRID_UNI['seq_daily'], GRID_UNI['lstm_units'],
    GRID_UNI['dropout_rate'], GRID_UNI['batch_size'],
    GRID_UNI['learning_rate']
))
total_uni = len(hp_uni_combos)

print(f"Memulai grid search univariate: {total_uni} kombinasi × {GRID_HORIZON} epoch")
print("=" * 72)

run_idx    = 0
start_time = datetime.now()

for (seq_d, units, drop, bs, lr) in hp_uni_combos:
    run_idx += 1
    X_tr = sequences_uni[seq_d]['X_train']
    y_tr = sequences_uni[seq_d]['y_train']

    tf.keras.backend.clear_session()
    gc.collect()

    model = build_lstm_univariate(
        seq_daily=seq_d, lstm_units=units,
        dropout_rate=drop, learning_rate=lr
    )
    history = model.fit(
        X_tr, y_tr,
        epochs=GRID_HORIZON, batch_size=bs,
        validation_split=VAL_SPLIT,
        verbose=0, shuffle=False
    )

    val_loss = min(history.history['val_loss'])
    config   = {
        "seq_daily": seq_d, "lstm_units": units,
        "dropout_rate": drop, "batch_size": bs,
        "learning_rate": lr,
        "val_loss_proxy": round(float(val_loss), 8),
    }
    grid_results_uni.append(config)

    marker = " ◄ BEST" if val_loss < best_uni['val_loss'] else ""
    if val_loss < best_uni['val_loss']:
        best_uni = {"val_loss": val_loss, "config": config}

    elapsed = (datetime.now() - start_time).seconds
    print(f"[{run_idx:2d}/{total_uni}] d={seq_d:2d} u={units:3d} dr={drop} "
          f"bs={bs:2d} lr={lr} → val_loss={val_loss:.6f}{marker}  ({elapsed}s)")

grid_results_uni_sorted = sorted(grid_results_uni, key=lambda x: x['val_loss_proxy'])
with open(MODEL_DIR / "grid_results_univariate.json", "w") as f:
    json.dump({"best": best_uni['config'],
               "all_results": grid_results_uni_sorted}, f, indent=2)

print(f"\n✓ Grid search univariate selesai")
print(f"  Best config   : {best_uni['config']}")
print(f"  Best val_loss : {best_uni['val_loss']:.6f}")
print(f"  Disimpan      → grid_results_univariate.json")


Memulai grid search univariate: 48 kombinasi × 5 epoch
[ 1/48] d=22 u= 64 dr=0.2 bs=16 lr=0.001 → val_loss=0.004496 ◄ BEST  (32s)
[ 2/48] d=22 u= 64 dr=0.2 bs=16 lr=0.0005 → val_loss=0.002907 ◄ BEST  (62s)
[ 3/48] d=22 u= 64 dr=0.2 bs=32 lr=0.001 → val_loss=0.001534 ◄ BEST  (86s)
[ 4/48] d=22 u= 64 dr=0.2 bs=32 lr=0.0005 → val_loss=0.001810  (111s)
[ 5/48] d=22 u= 64 dr=0.3 bs=16 lr=0.001 → val_loss=0.003368  (141s)
[ 6/48] d=22 u= 64 dr=0.3 bs=16 lr=0.0005 → val_loss=0.003849  (170s)
[ 7/48] d=22 u= 64 dr=0.3 bs=32 lr=0.001 → val_loss=0.004763  (195s)
[ 8/48] d=22 u= 64 dr=0.3 bs=32 lr=0.0005 → val_loss=0.006789  (220s)
[ 9/48] d=22 u=128 dr=0.2 bs=16 lr=0.001 → val_loss=0.002040  (268s)
[10/48] d=22 u=128 dr=0.2 bs=16 lr=0.0005 → val_loss=0.002830  (306s)
[11/48] d=22 u=128 dr=0.2 bs=32 lr=0.001 → val_loss=0.001614  (335s)
[12/48] d=22 u=128 dr=0.2 bs=32 lr=0.0005 → val_loss=0.001450 ◄ BEST  (375s)
[13/48] d=22 u=128 dr=0.3 bs=16 lr=0.001 → val_loss=0.004533  (422s)
[14/48] d=22 u=12

In [9]:
print("=" * 65)
print("RANGKUMAN GRID SEARCH")
print("=" * 65)
print("\n── MULTIVARIATE (hyperparameter arsitektur) ─────────────────")
for k, v in best_multi['config'].items():
    print(f"   {k:20s}: {v}")
print("\n── UNIVARIATE ───────────────────────────────────────────────")
for k, v in best_uni['config'].items():
    print(f"   {k:20s}: {v}")
print()
print("Top-5 multivariate configs (grid search proxy):")
for i, r in enumerate(grid_results_multi_sorted[:5], 1):
    print(f"  [{i}] d={r['seq_daily']:2d} m={r['seq_monthly']:2d} u={r['lstm_units']:3d} "
          f"dr={r['dropout_rate']} bs={r['batch_size']:2d} lr={r['learning_rate']} "
          f"→ val_loss={r['val_loss_proxy']:.6f}")


RANGKUMAN GRID SEARCH

── MULTIVARIATE (hyperparameter arsitektur) ─────────────────
   seq_daily           : 60
   seq_monthly         : 12
   lstm_units          : 128
   dropout_rate        : 0.2
   batch_size          : 32
   learning_rate       : 0.001
   val_loss_proxy      : 0.00129656

── UNIVARIATE ───────────────────────────────────────────────
   seq_daily           : 90
   lstm_units          : 128
   dropout_rate        : 0.3
   batch_size          : 32
   learning_rate       : 0.001
   val_loss_proxy      : 0.00119449

Top-5 multivariate configs (grid search proxy):
  [1] d=60 m=12 u=128 dr=0.2 bs=32 lr=0.001 → val_loss=0.001297
  [2] d=22 m= 6 u=128 dr=0.2 bs=32 lr=0.0005 → val_loss=0.001391
  [3] d=90 m=12 u=128 dr=0.3 bs=32 lr=0.001 → val_loss=0.001420
  [4] d=90 m=12 u=128 dr=0.2 bs=32 lr=0.001 → val_loss=0.001443
  [5] d=90 m= 3 u=128 dr=0.3 bs=32 lr=0.0005 → val_loss=0.001483


## 7. Ablation Study: `seq_monthly`

Proxy 5 epoch pada grid search belum cukup menangkap kontribusi branch bulanan, sebab data makroekonomi perlu lebih banyak epoch untuk memunculkan sinyal yang berarti. Karena itu `seq_monthly` ditentukan melalui ablation study tersendiri: tiap kandidat `seq_monthly` ∈ {3, 6, 12} dilatih penuh dengan hyperparameter arsitektur dikunci dari hasil grid search, lalu dipilih yang terbaik. Dengan begitu pemilihan `seq_monthly` berpijak pada evaluasi yang adil dan empiris, bukan pada proxy yang terbukti underestimate.

In [10]:
bc_m_arch = dict(best_multi['config'])
seq_d_best = bc_m_arch['seq_daily']

print(f"Hyperparameter arsitektur (dari grid search):")
print(f"  seq_daily    : {seq_d_best}")
print(f"  lstm_units   : {bc_m_arch['lstm_units']}")
print(f"  dropout_rate : {bc_m_arch['dropout_rate']}")
print(f"  batch_size   : {bc_m_arch['batch_size']}")
print(f"  learning_rate: {bc_m_arch['learning_rate']}")

print(f"\nMemulai ablation study seq_monthly ∈ {GRID_CONFIG['seq_monthly_candidates']}")
print(f"Setiap kandidat: 1 run × max {FINAL_EPOCHS} epoch (EarlyStopping patience={PATIENCE_ES})")
print("=" * 65)

ablation_results = []

for seq_m in GRID_CONFIG['seq_monthly_candidates']:
    key   = (seq_d_best, seq_m)
    seqs  = sequences_dual[key]

    X_d_tr = seqs['X_daily_train']
    X_m_tr = seqs['X_monthly_train']
    y_tr   = seqs['y_train']
    X_d_te = seqs['X_daily_test']
    X_m_te = seqs['X_monthly_test']
    y_te   = seqs['y_test']

    tf.random.set_seed(42)
    np.random.seed(42)
    tf.keras.backend.clear_session()
    gc.collect()

    ckpt_path = str(MODEL_DIR / f"ablation_multi_seqm{seq_m}.keras")
    model = build_lstm_dual_branch(
        seq_daily=seq_d_best, seq_monthly=seq_m,
        lstm_units=bc_m_arch['lstm_units'],
        dropout_rate=bc_m_arch['dropout_rate'],
        learning_rate=bc_m_arch['learning_rate'],
    )
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=PATIENCE_LR, min_lr=1e-7, verbose=0),
        ModelCheckpoint(filepath=ckpt_path, monitor='val_loss',
                        save_best_only=True, verbose=0),
    ]
    history = model.fit(
        [X_d_tr, X_m_tr], y_tr,
        epochs=FINAL_EPOCHS,
        batch_size=bc_m_arch['batch_size'],
        validation_split=VAL_SPLIT,
        callbacks=callbacks,
        shuffle=False, verbose=0
    )

    # Load checkpoint terbaik sebelum evaluasi
    model = tf.keras.models.load_model(ckpt_path)
    y_pred = model.predict([X_d_te, X_m_te], verbose=0).flatten()
    metrics = compute_metrics(y_te, y_pred, scaler_target)

    result = {
        "seq_monthly"   : seq_m,
        "best_val_loss" : float(min(history.history['val_loss'])),
        "epochs_ran"    : len(history.history['loss']),
        "rmse"          : float(metrics['rmse']),
        "mape"          : float(metrics['mape']),
        "r2"            : float(metrics['r2']),
        "checkpoint"    : ckpt_path,
    }
    ablation_results.append(result)
    print(f"  seq_monthly={seq_m:2d} | epochs={result['epochs_ran']:3d} "
          f"| val_loss={result['best_val_loss']:.6f} "
          f"| RMSE={result['rmse']:.2f} MAPE={result['mape']:.4f}% R²={result['r2']:.4f}")

print()
print("=" * 65)
print("ABLATION RESULTS:")
print(f"  {'seq_m':>6} {'RMSE':>8} {'MAPE':>8} {'R²':>8}")
print(f"  {'-'*35}")
for r in ablation_results:
    print(f"  {r['seq_monthly']:>6} {r['rmse']:>8.2f} {r['mape']:>8.4f} {r['r2']:>8.4f}")

# Pilih seq_monthly terbaik berdasarkan RMSE terkecil
best_ablation = min(ablation_results, key=lambda x: x['rmse'])
best_seq_monthly = best_ablation['seq_monthly']
print(f"\n✓ Best seq_monthly: {best_seq_monthly} "
      f"(RMSE={best_ablation['rmse']:.2f}, R²={best_ablation['r2']:.4f})")

with open(MODEL_DIR / "ablation_seq_monthly.json", "w") as f:
    json.dump({
        "seq_daily_fixed"  : seq_d_best,
        "arch_config"      : {k: v for k, v in bc_m_arch.items()
                               if k not in ['seq_monthly', 'val_loss_proxy']},
        "results"          : ablation_results,
        "best_seq_monthly" : best_seq_monthly,
    }, f, indent=2)
print(f"✓ Ablation results disimpan → ablation_seq_monthly.json")


Hyperparameter arsitektur (dari grid search):
  seq_daily    : 60
  lstm_units   : 128
  dropout_rate : 0.2
  batch_size   : 32
  learning_rate: 0.001

Memulai ablation study seq_monthly ∈ [3, 6, 12]
Setiap kandidat: 1 run × max 150 epoch (EarlyStopping patience=20)
  seq_monthly= 3 | epochs= 21 | val_loss=0.001636 | RMSE=212.41 MAPE=2.5857% R²=0.4829
  seq_monthly= 6 | epochs= 23 | val_loss=0.004744 | RMSE=619.01 MAPE=7.9732% R²=-3.3915
  seq_monthly=12 | epochs= 30 | val_loss=0.001341 | RMSE=734.71 MAPE=9.4211% R²=-5.1866

ABLATION RESULTS:
   seq_m     RMSE     MAPE       R²
  -----------------------------------
       3   212.41   2.5857   0.4829
       6   619.01   7.9732  -3.3915
      12   734.71   9.4211  -5.1866

✓ Best seq_monthly: 3 (RMSE=212.41, R²=0.4829)
✓ Ablation results disimpan → ablation_seq_monthly.json


## 8. Persiapan Data Final Training

Menyusun sekuens final dengan konfigurasi terpilih (hyperparameter dari grid search dan `seq_monthly` dari ablation) untuk tahap training akhir.

In [11]:
# ── Multivariate: gunakan best config dari grid search + best seq_monthly dari ablation ──
bc_m = dict(bc_m_arch)
bc_m['seq_monthly'] = best_seq_monthly

print("Final multivariate config:")
for k, v in bc_m.items():
    if k != 'val_loss_proxy':
        print(f"  {k:20s}: {v}")

key_m        = (bc_m['seq_daily'], bc_m['seq_monthly'])
seqs_m       = sequences_dual[key_m]
X_d_tr_final = seqs_m['X_daily_train']
X_m_tr_final = seqs_m['X_monthly_train']
y_tr_final   = seqs_m['y_train']
X_d_te_final = seqs_m['X_daily_test']
X_m_te_final = seqs_m['X_monthly_test']
y_te_final   = seqs_m['y_test']

print(f"\nTrain: X_daily={X_d_tr_final.shape}, X_monthly={X_m_tr_final.shape}, y={y_tr_final.shape}")
print(f"Test : X_daily={X_d_te_final.shape}, X_monthly={X_m_te_final.shape}, y={y_te_final.shape}")
print(f"y_te range: [{y_te_final.min():.4f}, {y_te_final.max():.4f}]")

# ── Univariate ──
bc_u = dict(best_uni['config'])
print(f"\nFinal univariate config:")
for k, v in bc_u.items():
    if k != 'val_loss_proxy':
        print(f"  {k:20s}: {v}")

X_tr_uni_final = sequences_uni[bc_u['seq_daily']]['X_train']
y_tr_uni_final = sequences_uni[bc_u['seq_daily']]['y_train']
X_te_uni_final = sequences_uni[bc_u['seq_daily']]['X_test']
y_te_uni_final = sequences_uni[bc_u['seq_daily']]['y_test']

print(f"\nTrain: X={X_tr_uni_final.shape}, y={y_tr_uni_final.shape}")
print(f"Test : X={X_te_uni_final.shape}, y={y_te_uni_final.shape}")
print(f"y_te range: [{y_te_uni_final.min():.4f}, {y_te_uni_final.max():.4f}]")


Final multivariate config:
  seq_daily           : 60
  seq_monthly         : 3
  lstm_units          : 128
  dropout_rate        : 0.2
  batch_size          : 32
  learning_rate       : 0.001

Train: X_daily=(1864, 60, 2), X_monthly=(1864, 3, 2), y=(1864,)
Test : X_daily=(485, 60, 2), X_monthly=(485, 3, 2), y=(485,)
y_te range: [0.7775, 1.1738]

Final univariate config:
  seq_daily           : 90
  lstm_units          : 128
  dropout_rate        : 0.3
  batch_size          : 32
  learning_rate       : 0.001

Train: X=(1849, 90, 1), y=(1849,)
Test : X=(485, 90, 1), y=(485,)
y_te range: [0.7775, 1.1738]


## 9. Final Training Multivariate (Multi-Seed)

Training diulang sebanyak `N_SEEDS` kali dengan random seed berbeda untuk meredam ketidakstabilan konvergensi LSTM yang dipicu oleh inisialisasi bobot acak (terutama pada CPU). Seed dengan `val_loss` terkecil dari checkpoint-nya dipakai sebagai model final.

In [12]:
seed_results_multi = []

print(f"Multi-seed final training multivariate — {N_SEEDS} seeds × max {FINAL_EPOCHS} epoch")
print("=" * 72)

for seed in SEED_LIST:
    print(f"\n[SEED {seed}] Training...", end=" ", flush=True)
    tf.random.set_seed(seed)
    np.random.seed(seed)
    tf.keras.backend.clear_session()
    gc.collect()

    ckpt_path = str(MODEL_DIR / f"ckpt_multi_seed{seed}.keras")
    model = build_lstm_dual_branch(
        seq_daily=bc_m['seq_daily'],
        seq_monthly=bc_m['seq_monthly'],
        lstm_units=bc_m['lstm_units'],
        dropout_rate=bc_m['dropout_rate'],
        learning_rate=bc_m['learning_rate'],
    )
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=PATIENCE_LR, min_lr=1e-7, verbose=0),
        ModelCheckpoint(filepath=ckpt_path, monitor='val_loss',
                        save_best_only=True, verbose=0),
    ]
    history = model.fit(
        [X_d_tr_final, X_m_tr_final], y_tr_final,
        epochs=FINAL_EPOCHS,
        batch_size=bc_m['batch_size'],
        validation_split=VAL_SPLIT,
        callbacks=callbacks,
        shuffle=False, verbose=0
    )
    best_val   = min(history.history['val_loss'])
    epochs_ran = len(history.history['loss'])
    seed_results_multi.append({
        "seed": seed, "best_val_loss": best_val,
        "epochs_ran": epochs_ran, "checkpoint_path": ckpt_path,
    })
    print(f"epochs={epochs_ran}, best_val_loss={best_val:.6f}")

best_seed_multi = min(seed_results_multi, key=lambda x: x['best_val_loss'])
print("\n" + "=" * 72)
print("SEED RESULTS MULTIVARIATE:")
for r in seed_results_multi:
    marker = " ◄ BEST" if r['seed'] == best_seed_multi['seed'] else ""
    print(f"  seed={r['seed']:3d} | val_loss={r['best_val_loss']:.6f} | epochs={r['epochs_ran']:3d}{marker}")
print(f"\n✓ Best seed: {best_seed_multi['seed']}  (val_loss={best_seed_multi['best_val_loss']:.6f})")


Multi-seed final training multivariate — 5 seeds × max 150 epoch

[SEED 42] Training... epochs=21, best_val_loss=0.003380

[SEED 7] Training... epochs=21, best_val_loss=0.002376

[SEED 13] Training... epochs=25, best_val_loss=0.005483

[SEED 21] Training... epochs=111, best_val_loss=0.002489

[SEED 99] Training... epochs=25, best_val_loss=0.003370

SEED RESULTS MULTIVARIATE:
  seed= 42 | val_loss=0.003380 | epochs= 21
  seed=  7 | val_loss=0.002376 | epochs= 21 ◄ BEST
  seed= 13 | val_loss=0.005483 | epochs= 25
  seed= 21 | val_loss=0.002489 | epochs=111
  seed= 99 | val_loss=0.003370 | epochs= 25

✓ Best seed: 7  (val_loss=0.002376)


In [13]:
tf.keras.backend.clear_session()
gc.collect()

model_multi = tf.keras.models.load_model(best_seed_multi['checkpoint_path'])
model_multi.save(MODEL_DIR / "best_multi_checkpoint.keras")
print(f"✓ Model multivariate loaded dari seed {best_seed_multi['seed']}")
print(f"✓ Disimpan ke best_multi_checkpoint.keras")


✓ Model multivariate loaded dari seed 7
✓ Disimpan ke best_multi_checkpoint.keras


## 10. Final Training Univariate (Multi-Seed)

Prosedur multi-seed yang sama diterapkan pada model univariate agar kedua model dibandingkan pada kondisi yang setara.

In [14]:
seed_results_uni = []

print(f"Multi-seed final training univariate — {N_SEEDS} seeds × max {FINAL_EPOCHS} epoch")
print("=" * 72)

for seed in SEED_LIST:
    print(f"\n[SEED {seed}] Training...", end=" ", flush=True)
    tf.random.set_seed(seed)
    np.random.seed(seed)
    tf.keras.backend.clear_session()
    gc.collect()

    ckpt_path = str(MODEL_DIR / f"ckpt_uni_seed{seed}.keras")
    model = build_lstm_univariate(
        seq_daily=bc_u['seq_daily'],
        lstm_units=bc_u['lstm_units'],
        dropout_rate=bc_u['dropout_rate'],
        learning_rate=bc_u['learning_rate'],
    )
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=PATIENCE_LR, min_lr=1e-7, verbose=0),
        ModelCheckpoint(filepath=ckpt_path, monitor='val_loss',
                        save_best_only=True, verbose=0),
    ]
    history = model.fit(
        X_tr_uni_final, y_tr_uni_final,
        epochs=FINAL_EPOCHS,
        batch_size=bc_u['batch_size'],
        validation_split=VAL_SPLIT,
        callbacks=callbacks,
        shuffle=False, verbose=0
    )
    best_val   = min(history.history['val_loss'])
    epochs_ran = len(history.history['loss'])
    seed_results_uni.append({
        "seed": seed, "best_val_loss": best_val,
        "epochs_ran": epochs_ran, "checkpoint_path": ckpt_path,
    })
    print(f"epochs={epochs_ran}, best_val_loss={best_val:.6f}")

best_seed_uni = min(seed_results_uni, key=lambda x: x['best_val_loss'])
print("\n" + "=" * 72)
print("SEED RESULTS UNIVARIATE:")
for r in seed_results_uni:
    marker = " ◄ BEST" if r['seed'] == best_seed_uni['seed'] else ""
    print(f"  seed={r['seed']:3d} | val_loss={r['best_val_loss']:.6f} | epochs={r['epochs_ran']:3d}{marker}")
print(f"\n✓ Best seed: {best_seed_uni['seed']}  (val_loss={best_seed_uni['best_val_loss']:.6f})")


Multi-seed final training univariate — 5 seeds × max 150 epoch

[SEED 42] Training... epochs=34, best_val_loss=0.001034

[SEED 7] Training... epochs=32, best_val_loss=0.001041

[SEED 13] Training... epochs=37, best_val_loss=0.000995

[SEED 21] Training... epochs=26, best_val_loss=0.001308

[SEED 99] Training... epochs=35, best_val_loss=0.001063

SEED RESULTS UNIVARIATE:
  seed= 42 | val_loss=0.001034 | epochs= 34
  seed=  7 | val_loss=0.001041 | epochs= 32
  seed= 13 | val_loss=0.000995 | epochs= 37 ◄ BEST
  seed= 21 | val_loss=0.001308 | epochs= 26
  seed= 99 | val_loss=0.001063 | epochs= 35

✓ Best seed: 13  (val_loss=0.000995)


In [15]:
tf.keras.backend.clear_session()
gc.collect()

model_uni = tf.keras.models.load_model(best_seed_uni['checkpoint_path'])
model_uni.save(MODEL_DIR / "best_uni_checkpoint.keras")
print(f"✓ Model univariate loaded dari seed {best_seed_uni['seed']}")
print(f"✓ Disimpan ke best_uni_checkpoint.keras")


✓ Model univariate loaded dari seed 13
✓ Disimpan ke best_uni_checkpoint.keras


## 11. Evaluasi Test Set

Kedua model dievaluasi pada periode test memakai RMSE, MAPE, dan R² untuk membandingkan akurasi prediksi multivariate terhadap baseline univariate.

In [16]:
# Prediksi dari best checkpoint
y_pred_multi_scaled = model_multi.predict([X_d_te_final, X_m_te_final], verbose=0).flatten()
y_pred_uni_scaled   = model_uni.predict(X_te_uni_final, verbose=0).flatten()

# Diagnostik skala
print("Diagnostik skala prediksi:")
print(f"  y_te_final (multi) : [{y_te_final.min():.4f}, {y_te_final.max():.4f}]  mean={y_te_final.mean():.4f}")
print(f"  pred_multi         : [{y_pred_multi_scaled.min():.4f}, {y_pred_multi_scaled.max():.4f}]  mean={y_pred_multi_scaled.mean():.4f}")
print(f"  y_te_uni_final     : [{y_te_uni_final.min():.4f}, {y_te_uni_final.max():.4f}]  mean={y_te_uni_final.mean():.4f}")
print(f"  pred_uni           : [{y_pred_uni_scaled.min():.4f}, {y_pred_uni_scaled.max():.4f}]  mean={y_pred_uni_scaled.mean():.4f}")

metrics_multi = compute_metrics(y_te_final,     y_pred_multi_scaled, scaler_target)
metrics_uni   = compute_metrics(y_te_uni_final, y_pred_uni_scaled,   scaler_uni)

print("\n" + "=" * 55)
print(f"{'Metrik':<12} {'Univariate':>15} {'Multivariate':>15}")
print("-" * 55)
print(f"{'RMSE':<12} {metrics_uni['rmse']:>15.2f} {metrics_multi['rmse']:>15.2f}")
print(f"{'MAPE (%)':<12} {metrics_uni['mape']:>15.4f} {metrics_multi['mape']:>15.4f}")
print(f"{'R²':<12} {metrics_uni['r2']:>15.4f} {metrics_multi['r2']:>15.4f}")
print("=" * 55)

delta_rmse = metrics_uni['rmse'] - metrics_multi['rmse']
delta_mape = metrics_uni['mape'] - metrics_multi['mape']
delta_r2   = metrics_multi['r2'] - metrics_uni['r2']
winner     = "multivariate" if delta_rmse > 0 else "univariate"

print(f"\nDelta (Multi − Uni):")
print(f"  RMSE : {delta_rmse:+.2f}  ({'Multivariate lebih baik' if delta_rmse > 0 else 'Univariate lebih baik'})")
print(f"  MAPE : {delta_mape:+.4f}% ({'Multivariate lebih baik' if delta_mape > 0 else 'Univariate lebih baik'})")
print(f"  R²   : {delta_r2:+.4f}  ({'Multivariate lebih baik' if delta_r2 > 0 else 'Univariate lebih baik'})")
print(f"\n→ Winner: {winner.upper()}")


Diagnostik skala prediksi:
  y_te_final (multi) : [0.7775, 1.1738]  mean=0.9297
  pred_multi         : [0.9041, 1.0836]  mean=0.9775
  y_te_uni_final     : [0.7775, 1.1738]  mean=0.9297
  pred_uni           : [0.8054, 1.0864]  mean=0.9149

Metrik            Univariate    Multivariate
-------------------------------------------------------
RMSE                  115.76          229.96
MAPE (%)              1.2267          2.8661
R²                    0.8464          0.3939

Delta (Multi − Uni):
  RMSE : -114.20  (Univariate lebih baik)
  MAPE : -1.6394% (Univariate lebih baik)
  R²   : -0.4525  (Univariate lebih baik)

→ Winner: UNIVARIATE


## 12. Simpan Artefak

Menyimpan model final, konfigurasi, dan metrik evaluasi ke `model/model_multivariate_v2/`.

In [17]:
# Model final
model_multi.save(MODEL_DIR / "model_multivariate.keras")
model_uni.save(MODEL_DIR / "model_univariate.keras")
print("✓ model_multivariate.keras")
print("✓ model_univariate.keras")

# Config lengkap
model_config = {
    "created_at"  : datetime.now().isoformat(),
    "version"     : "v2",
    "preprocessing_version": "v2",
    "notes"       : [
        "seq_monthly dipilih melalui ablation study full training, bukan grid search proxy",
        "Multi-seed training: best seed dipilih berdasarkan val_loss terkecil",
        "y_test konsisten (canonical) — tidak dipengaruhi seq_monthly"
    ],
    "multivariate": {
        "architecture"         : "Dual-Branch LSTM (Keras Functional API)",
        "hyperparameters"      : {k: v for k, v in bc_m.items()
                                  if k != 'val_loss_proxy'},
        "seq_monthly_selection": "ablation study",
        "ablation_results"     : ablation_results,
        "best_seed"            : best_seed_multi['seed'],
        "seed_results"         : seed_results_multi,
        "final_val_loss"       : best_seed_multi['best_val_loss'],
        "test_metrics"         : {
            "rmse": round(float(metrics_multi['rmse']), 4),
            "mape": round(float(metrics_multi['mape']), 4),
            "r2"  : round(float(metrics_multi['r2']),   4),
        },
        "input_daily_shape"    : list(X_d_te_final.shape[1:]),
        "input_monthly_shape"  : list(X_m_te_final.shape[1:]),
        "daily_features"       : DAILY_FEATURES,
        "monthly_features"     : MONTHLY_FEATURES,
        "target_col"           : TARGET_COL,
    },
    "univariate"  : {
        "architecture"  : "Single-Branch LSTM",
        "hyperparameters": {k: v for k, v in bc_u.items()
                            if k != 'val_loss_proxy'},
        "best_seed"     : best_seed_uni['seed'],
        "seed_results"  : seed_results_uni,
        "final_val_loss": best_seed_uni['best_val_loss'],
        "test_metrics"  : {
            "rmse": round(float(metrics_uni['rmse']), 4),
            "mape": round(float(metrics_uni['mape']), 4),
            "r2"  : round(float(metrics_uni['r2']),   4),
        },
        "input_shape"   : list(X_te_uni_final.shape[1:]),
        "target_col"    : TARGET_COL,
    },
    "comparison"  : {
        "delta_rmse": round(float(delta_rmse), 4),
        "delta_mape": round(float(delta_mape), 4),
        "delta_r2"  : round(float(delta_r2),   4),
        "winner"    : winner,
    }
}

with open(MODEL_DIR / "model_config.json", "w") as f:
    json.dump(model_config, f, indent=2)
print("✓ model_config.json")

# Predictions untuk visualisasi
dates_test_multi = sequences_dual[key_m]['dates_test']
dates_test_uni   = sequences_uni[bc_u['seq_daily']].get(
    'dates_test',
    [None] * len(y_te_uni_final)
)

predictions_data = {
    "multivariate": {
        "dates" : [str(pd.Timestamp(d).date()) if d is not None else None
                   for d in dates_test_multi],
        "y_true": metrics_multi['y_true'].tolist(),
        "y_pred": metrics_multi['y_pred'].tolist(),
    },
    "univariate"  : {
        "dates" : [str(pd.Timestamp(d).date()) if d is not None else None
                   for d in dates_test_uni],
        "y_true": metrics_uni['y_true'].tolist(),
        "y_pred": metrics_uni['y_pred'].tolist(),
    },
}
joblib.dump(predictions_data, MODEL_DIR / "predictions_test.joblib")
print("✓ predictions_test.joblib")

print()
print("=" * 60)
print(f"File tersimpan di: {MODEL_DIR}")
print("=" * 60)
for p in sorted(MODEL_DIR.iterdir()):
    print(f"  {p.name:45s} {p.stat().st_size/1024:8.1f} KB")


✓ model_multivariate.keras
✓ model_univariate.keras
✓ model_config.json
✓ predictions_test.joblib

File tersimpan di: d:\project_skripsi\model\model_multivariate_v2
  ablation_multi_seqm12.keras                     2851.7 KB
  ablation_multi_seqm3.keras                      2851.7 KB
  ablation_multi_seqm6.keras                      2851.7 KB
  ablation_seq_monthly.json                          1.1 KB
  best_multi_checkpoint.keras                     2851.3 KB
  best_uni_checkpoint.keras                       1429.7 KB
  ckpt_multi_seed13.keras                         2851.7 KB
  ckpt_multi_seed21.keras                         2851.7 KB
  ckpt_multi_seed42.keras                         2851.7 KB
  ckpt_multi_seed7.keras                          2851.7 KB
  ckpt_multi_seed99.keras                         2851.7 KB
  ckpt_uni_seed13.keras                           1430.0 KB
  ckpt_uni_seed21.keras                           1430.0 KB
  ckpt_uni_seed42.keras                           1430.

## 13. Ringkasan Akhir

Rangkuman konfigurasi terpilih beserta hasil akhir kedua model.

In [18]:
with open(MODEL_DIR / "model_config.json") as f:
    cfg = json.load(f)

print("=" * 60)
print("FINAL SUMMARY — MODELING v2")
print("=" * 60)
print(f"  Version              : {cfg['version']}")
print(f"  Preprocessing        : {cfg['preprocessing_version']}")
print()
print(f"  ── Multivariate ──────────────────────────────")
hp = cfg['multivariate']['hyperparameters']
print(f"  seq_daily            : {hp['seq_daily']}")
print(f"  seq_monthly          : {hp['seq_monthly']}  (dari ablation study)")
print(f"  lstm_units           : {hp['lstm_units']}")
print(f"  dropout_rate         : {hp['dropout_rate']}")
print(f"  best_seed            : {cfg['multivariate']['best_seed']}")
print(f"  RMSE                 : {cfg['multivariate']['test_metrics']['rmse']}")
print(f"  MAPE                 : {cfg['multivariate']['test_metrics']['mape']}%")
print(f"  R²                   : {cfg['multivariate']['test_metrics']['r2']}")
print()
print(f"  ── Univariate ────────────────────────────────")
hp_u = cfg['univariate']['hyperparameters']
print(f"  seq_daily            : {hp_u['seq_daily']}")
print(f"  lstm_units           : {hp_u['lstm_units']}")
print(f"  best_seed            : {cfg['univariate']['best_seed']}")
print(f"  RMSE                 : {cfg['univariate']['test_metrics']['rmse']}")
print(f"  MAPE                 : {cfg['univariate']['test_metrics']['mape']}%")
print(f"  R²                   : {cfg['univariate']['test_metrics']['r2']}")
print()
print(f"  ── Perbandingan ──────────────────────────────")
print(f"  ΔRMSE (Multi−Uni)    : {cfg['comparison']['delta_rmse']:+.4f}")
print(f"  ΔR²   (Multi−Uni)    : {cfg['comparison']['delta_r2']:+.4f}")
print(f"  Winner               : {cfg['comparison']['winner'].upper()}")
print("=" * 60)


FINAL SUMMARY — MODELING v2
  Version              : v2
  Preprocessing        : v2

  ── Multivariate ──────────────────────────────
  seq_daily            : 60
  seq_monthly          : 3  (dari ablation study)
  lstm_units           : 128
  dropout_rate         : 0.2
  best_seed            : 7
  RMSE                 : 229.9555
  MAPE                 : 2.8661%
  R²                   : 0.3939

  ── Univariate ────────────────────────────────
  seq_daily            : 90
  lstm_units           : 128
  best_seed            : 13
  RMSE                 : 115.7556
  MAPE                 : 1.2267%
  R²                   : 0.8464

  ── Perbandingan ──────────────────────────────
  ΔRMSE (Multi−Uni)    : -114.2000
  ΔR²   (Multi−Uni)    : -0.4525
  Winner               : UNIVARIATE
